# Titanic
## Score: .78468

In [44]:
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from pathlib import Path
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

SEED = 42
np.random.seed(SEED)

_root = Path.cwd()
_data = _root / "titanic"

RARE = {
    "Lady",
    "Countess",
    "Capt",
    "Col",
    "Don",
    "Dr",
    "Major",
    "Rev",
    "Sir",
    "Jonkheer",
    "Dona",
}


def build_xy(train_df: pd.DataFrame, test_df: pd.DataFrame):
    y = train_df["Survived"].to_numpy()
    tr = train_df.drop(columns=["Survived"]).copy()
    te = test_df.copy()
    ticket_vc = tr["Ticket"].value_counts()
    med_fare = tr["Fare"].median()
    mode_emb = tr["Embarked"].mode().iloc[0]
    tr["Fare"] = tr["Fare"].fillna(med_fare)
    te["Fare"] = te["Fare"].fillna(med_fare)
    tr["Embarked"] = tr["Embarked"].fillna(mode_emb)
    te["Embarked"] = te["Embarked"].fillna(mode_emb)

    def impute_age(ref: pd.DataFrame, df: pd.DataFrame) -> None:
        gmed = ref["Age"].median()
        for (sex, pcl), m in ref.groupby(["Sex", "Pclass"])["Age"].median().items():
            mm = m if pd.notna(m) else gmed
            msk = (df["Sex"] == sex) & (df["Pclass"] == pcl) & df["Age"].isna()
            df.loc[msk, "Age"] = mm
        df["Age"] = df["Age"].fillna(gmed)

    impute_age(tr, tr)
    impute_age(tr, te)

    def feats(df: pd.DataFrame) -> pd.DataFrame:
        out = df.copy()
        out["Title"] = out["Name"].str.extract(r",\s*([^\.]+)\.", expand=False).str.strip()
        out["Title"] = out["Title"].replace({"Mlle": "Miss", "Ms": "Miss", "Mme": "Mrs"})
        out["Title"] = out["Title"].where(~out["Title"].isin(RARE), "Rare")
        out["FamilySize"] = out["SibSp"] + out["Parch"] + 1
        out["IsAlone"] = (out["FamilySize"] == 1).astype(int)
        out["Deck"] = out["Cabin"].apply(
            lambda x: str(x)[0] if pd.notna(x) and str(x) and str(x)[0].isalpha() else "U"
        )
        out["TicketGroupSize"] = out["Ticket"].map(ticket_vc).fillna(1).astype(int)
        out["FarePerPerson"] = out["Fare"] / out["FamilySize"].clip(lower=1)
        out["HasCabin"] = out["Cabin"].notna().astype(int)
        out["AgeBin"] = pd.cut(
            out["Age"],
            bins=[0, 12, 18, 35, 60, 200],
            labels=["c", "t", "y", "a", "s"],
        ).astype(str)
        out = out.drop(columns=["Name", "Ticket", "Cabin", "PassengerId"], errors="ignore")
        out["Pclass"] = out["Pclass"].astype(str)
        for c in ["Sex", "Embarked", "Title", "Deck", "AgeBin"]:
            out[c] = out[c].astype(str)
        return out

    X = feats(tr)
    X_te = feats(te)
    return X, y, X_te


train_raw = pd.read_csv(_data / "train.csv")
test_raw = pd.read_csv(_data / "test.csv")
pid = test_raw["PassengerId"].values
X, y, X_test = build_xy(train_raw, test_raw)
cat_cols = ["Sex", "Embarked", "Title", "Deck", "Pclass", "AgeBin"]

X_tr, X_va, y_tr, y_va = train_test_split(
    X, y, test_size=0.12, random_state=SEED, stratify=y
)

clf = CatBoostClassifier(
    iterations=6000,
    learning_rate=0.03,
    depth=6,
    l2_leaf_reg=3.0,
    random_seed=SEED,
    thread_count=1,
    verbose=False,
)
clf.fit(
    X_tr,
    y_tr,
    cat_features=cat_cols,
    eval_set=(X_va, y_va),
    early_stopping_rounds=120,
    verbose=False,
)
va_acc = accuracy_score(y_va, clf.predict(X_va))
bi = clf.get_best_iteration()
if bi is None:
    bi = max(0, clf.tree_count_ - 1)

clf = CatBoostClassifier(
    iterations=bi + 1,
    learning_rate=0.03,
    depth=6,
    l2_leaf_reg=3.0,
    random_seed=SEED,
    thread_count=1,
    verbose=False,
)
clf.fit(X, y, cat_features=cat_cols, verbose=False)

pred = clf.predict(X_test)
out = pd.DataFrame({"PassengerId": pid, "Survived": pred.astype(int)})
out.to_csv(_root / "submission.csv", index=False)
print("val_acc", round(va_acc, 4), "trees", bi + 1, _root / "submission.csv")

val_acc 0.785 trees 90 c:\Users\ol1v3_7dwns5u\OneDrive\Desktop\ACTIVE\Titanic\submission.csv
